# 02. Business Performance and Customer Behavior Analysis

This notebook turns the cleaned transaction table into business KPIs and decision-oriented insights. It focuses on revenue, orders, customers, repeat purchase, new-versus-returning customer contribution, country performance, and product performance.

## 1. Setup and metric definitions

- **Revenue:** sum of `Quantity × UnitPrice` for valid completed purchases.
- **Orders:** distinct `InvoiceNo`.
- **Customers:** distinct known `CustomerID`.
- **Average order value (AOV):** total revenue divided by distinct orders.
- **Repeat customer:** customer with at least two distinct orders.
- **Repeat customer rate:** repeat customers divided by all purchasing customers.
- **New customer:** customer activity occurring in the customer's first purchase month.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.style.use("seaborn-v0_8-whitegrid")

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

clean_path = repo_root / "data" / "processed" / "online_retail_clean.csv.gz"
image_dir = repo_root / "images"
output_dir = repo_root / "data" / "processed"
image_dir.mkdir(parents=True, exist_ok=True)

if not clean_path.exists():
    raise FileNotFoundError("Run notebooks/01_data_cleaning.ipynb first.")

In [ ]:
df = pd.read_csv(
    clean_path,
    compression="gzip",
    parse_dates=["InvoiceDate"],
    dtype={"InvoiceNo": str, "StockCode": str, "CustomerID": str},
)
df["InvoiceMonth"] = df["InvoiceDate"].dt.to_period("M")
print(f"Loaded {len(df):,} cleaned transaction lines.")
display(df.head())

## 2. Executive KPI summary

In [ ]:
order_summary = (
    df.groupby("InvoiceNo", as_index=False)
      .agg(
          CustomerID=("CustomerID", "first"),
          InvoiceDate=("InvoiceDate", "min"),
          OrderRevenue=("Revenue", "sum"),
          OrderUnits=("Quantity", "sum"),
      )
)

customer_order_count = order_summary.groupby("CustomerID")["InvoiceNo"].nunique()
repeat_customers = int((customer_order_count >= 2).sum())
total_customers = int(customer_order_count.size)
repeat_rate = repeat_customers / total_customers

kpis = pd.Series({
    "Revenue_GBP": df["Revenue"].sum(),
    "Orders": order_summary["InvoiceNo"].nunique(),
    "Customers": total_customers,
    "Units_sold": df["Quantity"].sum(),
    "Average_order_value_GBP": order_summary["OrderRevenue"].mean(),
    "Orders_per_customer": len(order_summary) / total_customers,
    "Repeat_customers": repeat_customers,
    "Repeat_customer_rate": repeat_rate,
})
display(kpis.to_frame("value"))

## 3. Monthly business performance

December 2010 and December 2011 are partial months in the source data. They remain visible for transparency but should not be compared directly with full months.

In [ ]:
monthly = (
    df.groupby("InvoiceMonth")
      .agg(
          Revenue=("Revenue", "sum"),
          Orders=("InvoiceNo", "nunique"),
          ActiveCustomers=("CustomerID", "nunique"),
          Units=("Quantity", "sum"),
      )
      .reset_index()
)
monthly["AOV"] = monthly["Revenue"] / monthly["Orders"]
monthly["RevenueGrowth"] = monthly["Revenue"].pct_change()
monthly["MonthLabel"] = monthly["InvoiceMonth"].astype(str)
display(monthly)

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.plot(monthly["MonthLabel"], monthly["Revenue"], marker="o", linewidth=2.5, color="#2563EB")
ax1.set_ylabel("Revenue (GBP)", color="#2563EB")
ax1.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"£{x/1_000_000:.1f}M"))
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()
ax2.plot(monthly["MonthLabel"], monthly["Orders"], marker="s", linewidth=2, color="#F59E0B")
ax2.set_ylabel("Orders", color="#F59E0B")
ax2.grid(False)

plt.title("Monthly Revenue and Orders")
fig.tight_layout()
fig.savefig(image_dir / "01_monthly_revenue_orders.png", dpi=180, bbox_inches="tight")
plt.show()

## 4. New versus returning customer contribution

In [ ]:
first_purchase_month = df.groupby("CustomerID")["InvoiceMonth"].min()
df["FirstPurchaseMonth"] = df["CustomerID"].map(first_purchase_month)
df["CustomerType"] = np.where(
    df["InvoiceMonth"] == df["FirstPurchaseMonth"],
    "New",
    "Returning",
)

customer_type_monthly = (
    df.groupby(["InvoiceMonth", "CustomerType"])["Revenue"]
      .sum()
      .unstack(fill_value=0)
)
customer_type_monthly.index = customer_type_monthly.index.astype(str)
display(customer_type_monthly)

ax = customer_type_monthly.plot(
    kind="bar", stacked=True, figsize=(12, 6), color=["#60A5FA", "#1D4ED8"]
)
ax.set_title("Monthly Revenue from New and Returning Customers")
ax.set_xlabel("Invoice month")
ax.set_ylabel("Revenue (GBP)")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"£{x/1_000_000:.1f}M"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(image_dir / "02_new_returning_revenue.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. Geographic performance

The retailer is UK-based, so the chart below excludes the United Kingdom to reveal the most important international markets.

In [ ]:
country_summary = (
    df.groupby("Country")
      .agg(Revenue=("Revenue", "sum"), Orders=("InvoiceNo", "nunique"), Customers=("CustomerID", "nunique"))
      .reset_index()
)
country_summary["AOV"] = country_summary["Revenue"] / country_summary["Orders"]
country_summary = country_summary.sort_values("Revenue", ascending=False)
display(country_summary.head(15))

top_international = country_summary[country_summary["Country"] != "United Kingdom"].head(10).sort_values("Revenue")
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_international["Country"], top_international["Revenue"], color="#3B82F6")
ax.set_title("Top International Markets by Revenue")
ax.set_xlabel("Revenue (GBP)")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"£{x/1_000:.0f}K"))
plt.tight_layout()
plt.savefig(image_dir / "03_top_international_markets.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Product performance

A direct revenue ranking may be dominated by one-off wholesale purchases or administrative lines such as postage and manual adjustments. We therefore show the raw ranking for auditability and use a recurring-merchandise view for the portfolio chart.

In [ ]:
product_summary = (
    df.groupby(["StockCode", "Description"], dropna=False)
      .agg(Revenue=("Revenue", "sum"), Units=("Quantity", "sum"), Orders=("InvoiceNo", "nunique"))
      .reset_index()
      .sort_values("Revenue", ascending=False)
)
display(product_summary.head(10))

administrative_codes = {"POST", "DOT", "M", "BANK CHARGES", "AMAZONFEE", "CRUK", "D", "S"}
product_summary["IsAdministrative"] = product_summary["StockCode"].str.upper().isin(administrative_codes)
recurring_products = product_summary.loc[
    (~product_summary["IsAdministrative"]) & (product_summary["Orders"] >= 10)
].copy()
top_products = recurring_products.head(10).sort_values("Revenue")
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(top_products["Description"].str.title(), top_products["Revenue"], color="#F59E0B")
ax.set_title("Top 10 Recurring Merchandise Products by Revenue")
ax.set_xlabel("Revenue (GBP)")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"£{x/1_000:.0f}K"))
plt.tight_layout()
plt.savefig(image_dir / "04_top_products_revenue.png", dpi=180, bbox_inches="tight")
plt.show()

## 7. Customer value concentration

In [ ]:
customer_summary = (
    df.groupby("CustomerID")
      .agg(Revenue=("Revenue", "sum"), Orders=("InvoiceNo", "nunique"), Units=("Quantity", "sum"))
      .sort_values("Revenue", ascending=False)
)
customer_summary["AOV"] = customer_summary["Revenue"] / customer_summary["Orders"]
customer_summary["CumulativeRevenueShare"] = customer_summary["Revenue"].cumsum() / customer_summary["Revenue"].sum()
customer_summary["CumulativeCustomerShare"] = np.arange(1, len(customer_summary) + 1) / len(customer_summary)

top_20_cutoff = max(1, int(np.ceil(len(customer_summary) * 0.20)))
top_20_revenue_share = customer_summary.iloc[:top_20_cutoff]["Revenue"].sum() / customer_summary["Revenue"].sum()
display(customer_summary.head(10))
print(f"Top 20% of customers contribute {top_20_revenue_share:.1%} of revenue.")

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(customer_summary["CumulativeCustomerShare"], customer_summary["CumulativeRevenueShare"], color="#7C3AED", linewidth=2.5)
ax.axvline(0.20, color="#EF4444", linestyle="--", linewidth=1.5)
ax.axhline(top_20_revenue_share, color="#EF4444", linestyle="--", linewidth=1.5)
ax.set_title("Customer Revenue Concentration")
ax.set_xlabel("Cumulative share of customers")
ax.set_ylabel("Cumulative share of revenue")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.tight_layout()
plt.savefig(image_dir / "05_customer_revenue_concentration.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Export reusable summary tables

In [ ]:
monthly.to_csv(output_dir / "monthly_kpis.csv", index=False)
country_summary.to_csv(output_dir / "country_summary.csv", index=False)
product_summary.to_csv(output_dir / "product_summary.csv", index=False)
customer_summary.reset_index().to_csv(output_dir / "customer_summary.csv", index=False)

peak_full_month = monthly[monthly["MonthLabel"].between("2011-01", "2011-11")].nlargest(1, "Revenue").iloc[0]
uk_revenue_share = country_summary.loc[country_summary["Country"] == "United Kingdom", "Revenue"].iloc[0] / df["Revenue"].sum()
returning_revenue_share = df.loc[df["CustomerType"] == "Returning", "Revenue"].sum() / df["Revenue"].sum()

print("BUSINESS INSIGHT SNAPSHOT")
print(f"1. Revenue: £{kpis['Revenue_GBP']:,.0f}; orders: {int(kpis['Orders']):,}; AOV: £{kpis['Average_order_value_GBP']:,.2f}.")
print(f"2. Repeat customer rate: {repeat_rate:.1%} ({repeat_customers:,} of {total_customers:,} customers).")
print(f"3. Peak full month: {peak_full_month['MonthLabel']} with £{peak_full_month['Revenue']:,.0f} revenue.")
print(f"4. UK revenue share: {uk_revenue_share:.1%}.")
print(f"5. Returning-customer revenue share: {returning_revenue_share:.1%}.")
print(f"6. Top 20% customer revenue share: {top_20_revenue_share:.1%}.")

## Analytical interpretation

The descriptive analysis establishes where revenue comes from and whether growth depends on acquisition or repeat purchasing. The next notebook will build a cohort matrix to measure how quickly customers return after their first purchase.